In [1]:
from __future__ import annotations

from collections import OrderedDict
from typing import Any

import pandas as pd
import torch


def tensor_shape(x: Any):
    if isinstance(x, torch.Tensor):
        return tuple(x.shape)
    if isinstance(x, (list, tuple)) and x and isinstance(x[0], torch.Tensor):
        return [tuple(t.shape) for t in x]
    return type(x).__name__


def parse_cft(shape: tuple[int, ...] | list[tuple[int, ...]] | str):
    if not isinstance(shape, tuple):
        return None, None, None, None
    if len(shape) == 4:
        b, c, f, t = shape
        return b, c, f, t
    if len(shape) == 3:
        b, cf, t = shape
        return b, cf, None, t
    if len(shape) == 2:
        b, d = shape
        return b, d, None, None
    return None, None, None, None


def inspect_model(model_name: str = "b6", train_type: str = "ptn", dataset: str = "vox2"):
    model = torch.hub.load(
        "IDRnD/ReDimNet",
        "ReDimNet",
        model_name=model_name,
        train_type=train_type,
        dataset=dataset,
    )
    model.eval()

    x = torch.zeros(1, 16000)
    traces: "OrderedDict[str, dict[str, Any]]" = OrderedDict()
    handles = []

    def add_hook(name: str, module):
        def hook(_module, inputs, output):
            in_shape = tensor_shape(inputs[0] if inputs else None)
            out_shape = tensor_shape(output)
            b, c, f, t = parse_cft(out_shape)
            traces[name] = {
                "module": type(_module).__name__,
                "in_shape": in_shape,
                "out_shape": out_shape,
                "B": b,
                "C_or_CF": c,
                "F": f,
                "T": t,
                "C_times_F": None if (c is None or f is None) else c * f,
                "note": "[B,C,F,T]" if f is not None else ("[B,CF,T] or [B,D]" if c is not None else "other"),
            }

        handles.append(module.register_forward_hook(hook))

    add_hook("spec", model.spec)
    add_hook("backbone.stem", model.backbone.stem)
    add_hook("backbone.stage0", model.backbone.stage0)
    add_hook("backbone.stage1", model.backbone.stage1)
    add_hook("backbone.stage2", model.backbone.stage2)
    add_hook("backbone.stage3", model.backbone.stage3)
    add_hook("backbone.stage4", model.backbone.stage4)
    add_hook("backbone.stage5", model.backbone.stage5)
    add_hook("backbone.fin_wght1d", model.backbone.fin_wght1d)
    add_hook("backbone.mfa", model.backbone.mfa)
    add_hook("backbone.fin_to2d", model.backbone.fin_to2d)
    add_hook("pool", model.pool)
    add_hook("bn", model.bn)
    add_hook("linear", model.linear)

    with torch.no_grad():
        _ = model(x)

    for handle in handles:
        handle.remove()

    rows = []
    for name, info in traces.items():
        rows.append(
            {
                "layer": name,
                "module": info["module"],
                "out_shape": info["out_shape"],
                "B": info["B"],
                "C_or_CF": info["C_or_CF"],
                "F": info["F"],
                "C*F": info["C_times_F"],
                "T": info["T"],
                "note": info["note"],
            }
        )

    df = pd.DataFrame(rows)
    display(df)

    print(f"MODEL: {model_name}")
    print("\n=== How to read C * F ===")
    print("For 4D tensors [B, C, F, T]:")
    print("  C = number of channels")
    print("  F = number of frequency groups / feature bins inside each channel")
    print("  T = time frames")
    print("  C * F = total feature rows if you flatten channels and frequency groups together")
    print("\nThis is why a later visualization may show stage4 as (C*F, T) = (2304, T) if stage4 is (192, 12, T).")

    print("\n=== Backbone stages only ===")
    stage_df = df[df["layer"].str.startswith("backbone.stage") | df["layer"].eq("backbone.stem")].copy()
    display(stage_df[["layer", "out_shape", "C_or_CF", "F", "C*F", "T"]])

    print("\n=== Stage4 interpretation ===")
    s4 = df[df["layer"] == "backbone.stage4"]
    if not s4.empty:
        row = s4.iloc[0]
        print(f"stage4 out_shape = {row['out_shape']}")
        print(f"stage4 has C = {row['C_or_CF']}, F = {row['F']}, T = {row['T']}")
        print(f"So flattening stage4 over C and F gives {row['C*F']} feature rows per time index.")


inspect_model(model_name="b6")


Using cache found in /home/SpeakerRec/.cache/torch/hub/IDRnD_ReDimNet_master


,layer,module,out_shape,B,C_or_CF,F,C*F,T,note
0,spec,MelBanks,"(1, 72, 67)",1,72,None,None,67.0,"[B,CF,T] or [B,D]"
1,backbone.stem,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
2,backbone.stage0,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
3,backbone.stage1,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
4,backbone.stage2,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
5,backbone.stage3,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
6,backbone.stage4,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
7,backbone.stage5,Sequential,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
8,backbone.fin_wght1d,weigth1d,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"
9,backbone.fin_to2d,Identity,"(1, 2304, 67)",1,2304,None,None,67.0,"[B,CF,T] or [B,D]"


MODEL: b6

=== How to read C * F ===
For 4D tensors [B, C, F, T]:
  C = number of channels
  F = number of frequency groups / feature bins inside each channel
  T = time frames
  C * F = total feature rows if you flatten channels and frequency groups together

This is why a later visualization may show stage4 as (C*F, T) = (2304, T) if stage4 is (192, 12, T).

=== Backbone stages only ===


,layer,out_shape,C_or_CF,F,C*F,T
1,backbone.stem,"(1, 2304, 67)",2304,None,None,67.0
2,backbone.stage0,"(1, 2304, 67)",2304,None,None,67.0
3,backbone.stage1,"(1, 2304, 67)",2304,None,None,67.0
4,backbone.stage2,"(1, 2304, 67)",2304,None,None,67.0
5,backbone.stage3,"(1, 2304, 67)",2304,None,None,67.0
6,backbone.stage4,"(1, 2304, 67)",2304,None,None,67.0
7,backbone.stage5,"(1, 2304, 67)",2304,None,None,67.0



=== Stage4 interpretation ===
stage4 out_shape = (1, 2304, 67)
stage4 has C = 2304, F = None, T = 67.0
So flattening stage4 over C and F gives None feature rows per time index.
